# Chutes Paved Path (Single Way)

This notebook verifies the one and only way we call Chutes (OpenAI-compatible):
- Provider: `openai_like`
- Auth: `Authorization: Bearer $CHUTES_API_KEY` (never x-api-key)
- Model: exactly one pinned vendor id via `CHUTES_TEXT_MODEL`
- JSON mode: `response_format={"type":"json_object"}`
- Lifecycle: reuse a Router, then close it


In [ ]:
import os, asyncio, json
from litellm import Router

base = os.environ['CHUTES_API_BASE']
key  = os.environ['CHUTES_API_KEY']
mid  = os.environ['CHUTES_TEXT_MODEL']

model_list = [{
    'model_name': 'chutes/text',
    'litellm_params': {
        'model': mid,
        'custom_llm_provider': 'openai_like',
        'api_base': base,
        'api_key': None,
        'extra_headers': {'Authorization': f'Bearer {key}'},
    },
}]

router = Router(model_list=model_list, num_retries=0, default_litellm_params={'timeout': 20})

async def main():
    out = await router.acompletion(
        model='chutes/text',
        messages=[
            {'role':'system','content':'Return only {\"ok\":true} as JSON.'},
            {'role':'user','content':'ping'},
        ],
        response_format={'type':'json_object'},
        temperature=0,
        timeout=20,
    )
    c = out.choices[0].message.get('content')
    print('ok:', c if isinstance(c,str) else json.dumps(c))
    await router.aclose()

await main()
